# 📰 Get Posts - WP Rest API
Notebook ini mengambil semua **posts** dari WordPress menggunakan WordPress REST API dengan autentikasi Basic Auth.

Link to Google Colab: [here](https://colab.research.google.com/drive/1uBjU8vAGBjysxzwanqqD_gelR7NU4wT6?usp=sharing)

In [ ]:
import requests
import csv
from requests.auth import HTTPBasicAuth

# Jika menggunakan Google Colab
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

In [ ]:
# Endpoint Posts
wordpress_url = "https://www.balitouristic.com/wp-json/wp/v2/posts"

# Ambil kredensial
if userdata:
    username = userdata.get('admin_bali')
    password = userdata.get('pass_bali')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("Kredensial tidak ditemukan di userdata.")

auth = HTTPBasicAuth(username, password)
print("✅ Kredensial berhasil dimuat.")

In [ ]:
# Fungsi ambil semua posts dengan progress
def fetch_all_posts(url, auth):
    all_posts = []
    page = 1

    print("🚀 Memulai pengambilan posts...\n")

    while True:
        print(f"📄 Mengambil halaman {page} ...")

        response = requests.get(
            url,
            auth=auth,
            params={
                "per_page": 100,
                "page": page,
                "_fields": "id,date,modified,slug,status,title,content,excerpt,author,featured_media,categories,tags,link"
            }
        )

        if response.status_code == 200:
            posts = response.json()

            if not posts:
                print("✅ Tidak ada data lagi. Selesai.\n")
                break

            print(f"   ➜ Ditemukan {len(posts)} post di halaman {page}")

            all_posts.extend(posts)
            page += 1

        elif response.status_code == 400:
            # WordPress mengembalikan 400 jika halaman melebihi total
            print("✅ Semua halaman telah diambil. Selesai.\n")
            break

        else:
            print(f"❌ Gagal di halaman {page}")
            print("Status Code:", response.status_code)
            print(response.text)
            break

    print(f"\n🎯 Total post berhasil diambil: {len(all_posts)}\n")
    return all_posts

In [ ]:
# Ambil semua posts
posts = fetch_all_posts(wordpress_url, auth)

In [ ]:
# Preview data pertama
if posts:
    print("🔍 Preview post pertama:")
    import json
    print(json.dumps(posts[0], indent=2, ensure_ascii=False)[:1000], "...")
else:
    print("⚠ Tidak ada post ditemukan.")

In [ ]:
# Simpan ke CSV
if posts:
    csv_filename = "wordpress_posts.csv"

    with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        # Header
        writer.writerow([
            "ID",
            "Date",
            "Modified",
            "Slug",
            "Status",
            "Title",
            "Excerpt",
            "Author",
            "Featured Media",
            "Categories",
            "Tags",
            "Link"
        ])

        for post in posts:
            writer.writerow([
                post.get("id", ""),
                post.get("date", ""),
                post.get("modified", ""),
                post.get("slug", ""),
                post.get("status", ""),
                post["title"]["rendered"] if "title" in post else "",
                post["excerpt"]["rendered"] if "excerpt" in post else "",
                post.get("author", ""),
                post.get("featured_media", ""),
                ", ".join(str(c) for c in post.get("categories", [])),
                ", ".join(str(t) for t in post.get("tags", [])),
                post.get("link", "")
            ])

    print(f"💾 File CSV berhasil disimpan sebagai '{csv_filename}'")

    if IN_COLAB:
        files.download(csv_filename)

else:
    print("⚠ Tidak ada post ditemukan.")